In [1]:
import json
import re

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

train_data = []
validation_data = []
test_data = []
with open("../../backend/data/data-v1/train-v1.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line))
with open("../../backend/data/data-v1/validation-v1.jsonl", "r", encoding="utf8") as f:
    for line in f:
        validation_data.append(json.loads(line))
with open("../../backend/data/data-v1/test-v1.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line))

In [2]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r"[.!?,;:]", "", text)
    return text.split()

In [4]:
from collections import Counter

counter = Counter()
for item in train_data:
    tokens = tokenizer(item["transcript"])
    counter.update(tokens)

In [5]:
max_vocab_size = 2000
vocab = {"<pad>": 0, "<unk>": 1}

for word, _ in counter.most_common(max_vocab_size - 2):
    vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 873


In [6]:
def numericalizer(text):
    tokens = tokenizer(text)
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]


text = "Con bị ho, sổ mũi liên tục. Nhai lâu nuốt chậm lắm, ăn ít cơm thôi. Tè dầm, ngủ ngắn. Nặn đất không tập trung. Nhõng nhẽo cả ngày."
numericalized = numericalizer(text)
print(numericalized)

[31, 48, 65, 115, 103, 221, 215, 134, 107, 87, 100, 117, 2, 58, 14, 267, 120, 121, 3, 64, 131, 130, 9, 22, 23, 72, 73, 159, 175]


In [7]:
label_to_id = {
    "O": 0,
    "B-MEALS": 1,
    "I-MEALS": 2,
    "B-SLEEPING": 3,
    "I-SLEEPING": 4,
    "B-HEALTH": 5,
    "I-HEALTH": 6,
    "B-ACTIVITIES": 7,
    "I-ACTIVITIES": 8,
    "B-MOOD": 9,
    "I-MOOD": 10,
}

id_to_label = {value: key for key, value in label_to_id.items()}

In [8]:
def find_phrase(tokens, phrase_tokens, start_from=0):
    for i in range(start_from, len(tokens) - len(phrase_tokens) + 1):
        if tokens[i : i + len(phrase_tokens)] == phrase_tokens:
            return i
    return None

In [9]:
def create_bio_labels(transcript, output):
    tokens = tokenizer(transcript)

    bio_labels = ["O"] * len(tokens)

    label_field = [
        ("meals", "MEALS"),
        ("sleeping", "SLEEPING"),
        ("health", "HEALTH"),
        ("activities", "ACTIVITIES"),
        ("mood", "MOOD"),
    ]

    for field_name, entity_name in label_field:
        target = output.get(field_name)

        if target is None:
            continue

        target = target.strip()

        if target == "":
            continue

        phrases = [phrase.strip() for phrase in target.split(",") if phrase.strip()]

        search_position = 0

        for phrase in phrases:
            phrase_tokens = tokenizer(phrase)

            if not phrase_tokens:
                continue

            start = find_phrase(tokens, phrase_tokens, start_from=search_position)

            # Không tìm thấy exact span → bỏ qua
            if start is None:
                continue

            end = start + len(phrase_tokens)

            for i in range(start, end):
                if i == start:
                    bio_labels[i] = f"B-{entity_name}"
                else:
                    bio_labels[i] = f"I-{entity_name}"

            search_position = end

    return tokens, bio_labels

In [10]:
class BIODataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):

        item = self.dataset[index]

        transcript = item["transcript"]
        output = item["output"]

        _tokens, bio_labels = create_bio_labels(transcript, output)

        input_ids = numericalizer(transcript)

        label_ids = [label_to_id[label] for label in bio_labels]

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(label_ids, dtype=torch.long),
        )

In [11]:
def collate_fn(batch):

    texts, labels = zip(*batch)

    texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])

    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return texts, labels

In [12]:
train_dataset = BIODataset(train_data)
validation_dataset = BIODataset(validation_data)
test_dataset = BIODataset(test_data)

In [13]:
train_dataloader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn
)

validation_dataloader = DataLoader(
    validation_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn
)

test_dataloader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn
)

In [14]:
embedding_dim = 128

embedding = nn.Embedding(len(vocab), embedding_dim, padding_idx=vocab["<pad>"])

In [15]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        embedded = self.embedding(x)

        output, _ = self.lstm(embedded)

        output = self.fc(output)

        return output

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = LSTMClassifier(
    vocab_size=len(vocab), embedding_dim=embedding_dim, hidden_dim=64, num_classes=11
).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(
    model.parameters(),
)

cuda


In [17]:
num_epoch = 20

for epoch in range(num_epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for texts, labels in train_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        output = model(texts)
        loss = criterion(output.view(-1, output.size(-1)), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predictions = output.argmax(dim=-1)
        mask = labels != -100
        correct += (predictions[mask] == labels[mask]).sum().item()
        total += mask.sum().item()
    avg_loss = total_loss / len(train_dataloader)
    accuracy = correct / total * 100
    print(
        f"Epoch {epoch + 1}/{num_epoch} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch 1/20 | Loss: 1.8474 | Accuracy: 48.80%
Epoch 2/20 | Loss: 0.8495 | Accuracy: 79.74%
Epoch 3/20 | Loss: 0.4429 | Accuracy: 89.91%
Epoch 4/20 | Loss: 0.2941 | Accuracy: 92.80%
Epoch 5/20 | Loss: 0.2306 | Accuracy: 93.92%
Epoch 6/20 | Loss: 0.1960 | Accuracy: 94.58%
Epoch 7/20 | Loss: 0.1728 | Accuracy: 95.12%
Epoch 8/20 | Loss: 0.1562 | Accuracy: 95.48%
Epoch 9/20 | Loss: 0.1428 | Accuracy: 95.82%
Epoch 10/20 | Loss: 0.1323 | Accuracy: 96.05%
Epoch 11/20 | Loss: 0.1228 | Accuracy: 96.33%
Epoch 12/20 | Loss: 0.1150 | Accuracy: 96.68%
Epoch 13/20 | Loss: 0.1082 | Accuracy: 96.83%
Epoch 14/20 | Loss: 0.1028 | Accuracy: 96.94%
Epoch 15/20 | Loss: 0.0982 | Accuracy: 97.10%
Epoch 16/20 | Loss: 0.0913 | Accuracy: 97.38%
Epoch 17/20 | Loss: 0.0864 | Accuracy: 97.52%
Epoch 18/20 | Loss: 0.0817 | Accuracy: 97.67%
Epoch 19/20 | Loss: 0.0794 | Accuracy: 97.77%
Epoch 20/20 | Loss: 0.0761 | Accuracy: 97.84%


In [19]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for texts, labels in validation_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)

        predictions = outputs.argmax(dim=-1)

        mask = labels != -100

        correct += (predictions[mask] == labels[mask]).sum().item()

        total += mask.sum().item()

accuracy = correct / total

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.9036


In [20]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for texts, labels in test_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)

        predictions = outputs.argmax(dim=-1)

        mask = labels != -100

        correct += (predictions[mask] == labels[mask]).sum().item()

        total += mask.sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9640


In [23]:
id_to_vocab = {idx: word for word, idx in vocab.items()}

In [ ]:
import random

model.eval()

with torch.no_grad():
    index = random.randrange(len(validation_dataset))

    text, labels = validation_dataset[index]

    text = text.unsqueeze(0).to(device)
    labels = labels.unsqueeze(0).to(device)

    outputs = model(text)

    predictions = outputs.argmax(dim=-1)

    print(f"===== SAMPLE {index} =====")

    for token_id, true_id, pred_id in zip(text[0], labels[0], predictions[0]):
        if true_id == -100:
            continue

        token = id_to_vocab[token_id.item()]

        true_label = id_to_label[true_id.item()]
        pred_label = id_to_label[pred_id.item()]

        print(f"{token:15} True: {true_label:15} Pred: {pred_label}")

In [25]:
# === SAVE MODEL ===

import os

model_path = "../models/bio_lstm_model.pt"

os.makedirs(os.path.dirname(model_path), exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "vocab": vocab,
        "label_to_id": label_to_id,
        "embedding_dim": embedding_dim,
        "hidden_dim": 64,
        "num_classes": len(label_to_id),
    },
    model_path,
)

print(f"Model saved to: {model_path}")

Model saved to: ../models/bio_lstm_model.pt
